# τ(β) Crossing Test — Power-Law Spekter

**Spørsmål:** For p_i ∝ i^{-β}, kryssar τ(β) grensene e^{-γ} ≈ 0.5615 og 1/ζ(3) ≈ 0.8319 ved β=1 og β=3?

**Konjektur A:** τ(β=1) ≈ e^{-γ} (Zipf, harmonisk-tal-asymptotikk gir γ)

**Konjektur B:** τ(β=3) ≈ 1/ζ(3) (Apéry, partisjonsfunksjonen er ζ(3))

**Ingen GPU nødvendig — ren numpy.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import zeta

EULER_MASCHERONI = 0.5772156649015328
APERY            = 1.2020569031595942
TAU_MIN = np.exp(-EULER_MASCHERONI)  # 0.5615
TAU_MAX = 1.0 / APERY                # 0.8319

print(f"Goldilocks: [{TAU_MIN:.6f}, {TAU_MAX:.6f}]")
print(f"Testar konjektur: τ(β=1) ≈ {TAU_MIN:.4f}, τ(β=3) ≈ {TAU_MAX:.4f}")

In [ ]:
def compute_tau_beta(beta, r=1000):
    """Reknar tau for power-law spekter p_i ∝ i^{-beta}, i=1..r"""
    i = np.arange(1, r + 1, dtype=np.float64)
    weights = i ** (-beta)
    p = weights / weights.sum()
    # Shannon entropi
    H = -np.sum(p * np.log(p + 1e-300))
    r_eff = np.exp(H)
    tau = r_eff / r  # r_max = r (antal dimensjonar)
    return tau, r_eff, H

# Test ved beta=1 og beta=3
for beta in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
    tau, r_eff, H = compute_tau_beta(beta)
    marker = ""
    if abs(tau - TAU_MIN) < 0.02: marker = "  ← NÆR τ_min!"
    if abs(tau - TAU_MAX) < 0.02: marker = "  ← NÆR τ_max!"
    print(f"β={beta:.1f}: τ={tau:.6f}  r_eff={r_eff:.1f}  H={H:.4f}{marker}")

In [ ]:
# Finmasket skann: finn eksakt kvar τ(β) kryssar grensene
betas = np.linspace(0.1, 5.0, 500)
taus  = [compute_tau_beta(b)[0] for b in betas]

# Finn kryssingar
taus_arr = np.array(taus)
cross_min = betas[np.argmin(np.abs(taus_arr - TAU_MIN))]
cross_max = betas[np.argmin(np.abs(taus_arr - TAU_MAX))]
val_at_min = taus_arr[np.argmin(np.abs(taus_arr - TAU_MIN))]
val_at_max = taus_arr[np.argmin(np.abs(taus_arr - TAU_MAX))]

print(f"τ(β) ≈ τ_min={TAU_MIN:.4f} ved β ≈ {cross_min:.3f}  (τ={val_at_min:.6f})")
print(f"τ(β) ≈ τ_max={TAU_MAX:.4f} ved β ≈ {cross_max:.3f}  (τ={val_at_max:.6f})")
print(f"\nKonjektur A (β=1): {'STØTTA ✓' if abs(cross_min - 1.0) < 0.1 else f'IKKJE støtta — faktisk β={cross_min:.3f}'}")
print(f"Konjektur B (β=3): {'STØTTA ✓' if abs(cross_max - 3.0) < 0.1 else f'IKKJE støtta — faktisk β={cross_max:.3f}'}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(betas, taus, 'steelblue', linewidth=2, label='τ(β) = exp(H)/r')
ax.axhspan(TAU_MIN, TAU_MAX, alpha=0.12, color='green', label=f'Goldilocks [{TAU_MIN:.4f}, {TAU_MAX:.4f}]')
ax.axhline(TAU_MIN, color='green', linestyle='--', linewidth=1.2)
ax.axhline(TAU_MAX, color='red',   linestyle='--', linewidth=1.2)
ax.axvline(1.0, color='purple', linestyle=':', linewidth=1.5, label='β=1 (Zipf, γ-konjektur)')
ax.axvline(3.0, color='orange', linestyle=':', linewidth=1.5, label='β=3 (ζ(3)-konjektur)')
ax.axvline(cross_min, color='green', linestyle='-', linewidth=1, alpha=0.5, label=f'Faktisk τ_min-kryssing β={cross_min:.2f}')
ax.axvline(cross_max, color='red',   linestyle='-', linewidth=1, alpha=0.5, label=f'Faktisk τ_max-kryssing β={cross_max:.2f}')

ax.set_xlabel("β (power-law eksponent)", fontsize=12)
ax.set_ylabel("τ = exp(H) / r", fontsize=12)
ax.set_title("τ(β) for p_i ∝ i^{-β} — kryssar Goldilocks ved β=1 og β=3?", fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("tau_beta_crossing.png", dpi=150, bbox_inches='tight')
plt.show()
print("Lagra: tau_beta_crossing.png")

In [ ]:
print("=" * 60)
print("KONJEKTUR-TEST: power-law spekter og Goldilocks")
print("=" * 60)
print(f"\nτ_min = e^(-γ) = {TAU_MIN:.6f}")
print(f"Kryssing ved β = {cross_min:.4f}")
print(f"Avstand frå β=1: {abs(cross_min-1.0):.4f}")
print(f"\nτ_max = 1/ζ(3) = {TAU_MAX:.6f}")
print(f"Kryssing ved β = {cross_max:.4f}")
print(f"Avstand frå β=3: {abs(cross_max-3.0):.4f}")
print("\n" + "=" * 60)
a_støtta = abs(cross_min - 1.0) < 0.05
b_støtta = abs(cross_max - 3.0) < 0.05
if a_støtta and b_støtta:
    print("Begge konjekturar STØTTA — Goldilocks = power-law β∈[1,3]")
elif a_støtta:
    print(f"Konjektur A støtta (γ). Konjektur B IKKJE støtta — β={cross_max:.2f} ≠ 3")
elif b_støtta:
    print(f"Konjektur B støtta (ζ(3)). Konjektur A IKKJE støtta — β={cross_min:.2f} ≠ 1")
else:
    print(f"Ingen konjektur støtta. Faktiske kryssingar: β={cross_min:.2f} og β={cross_max:.2f}")
    print("Power-law-spekter forklarar IKKJE Goldilocks-grensene.")
print("=" * 60)